# Notebook: Evaluating Relevance in a Clinical Setting

This notebook demonstrates a sample workflow for evaluating the relevance of Large Language Model (LLM) responses to clinical queries. Following the MedHELM-inspired approach, we illustrate how to:

1. **Candidate LLMs**: Consider multiple model candidates (such as `gpt-4o`, `o1`, `phi-3`, `phi-4`).
2. **Task Categorization**: Choose a task from a clinical taxonomy (e.g., "Administration and Workflow" / "Prior Authorization" scenario).
3. **Model Evaluation**: Evaluate multiple dimensions (e.g., Relevance, Fluency, etc.) but dive deeper into Relevance.
4. **Evaluators**: Use both a **deterministic** approach and **LLM-as-a-Judge** approach for scoring Relevance.
5. **Generate Leaderboard**: Present results in a structured table (potentially using Azure AI Foundry).

We will leverage a set of sample queries about prior authorization criteria for Adalimumab in Crohn’s disease, comparing them with chunks from a policy document. 



In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

import os
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential

# Define the model configurations with environment variables for endpoints and API keys. Can add more as required.
models = [
    {
        "name": "gpt-4o",
        "endpoint": os.getenv("AZURE_OPENAI_ENDPOINT"),      # e.g., "https://your-resource.openai.azure.com/openai/deployments/gpt-4o"
        "api_key": os.getenv("AZURE_OPENAI_KEY")
    },
    # {
    #     "name": "o1",
    #     "endpoint": os.getenv("AZURE_OPENAI_ENDPOINT"),          # e.g., "https://your-o1-endpoint.com/your-deployment"
    #     "api_key": os.getenv("AZURE_OPENAI_KEY")
    # },
    # {
    #     "name": "phi-4",
    #     "endpoint": os.getenv("AZURE_PHI4_ENDPOINT"),        # e.g., "https://your-resource.azure.com/openai/deployments/phi-4"
    #     "api_key": os.getenv("AZURE_PHI4_API_KEY")
    # }
]

# Initialize the ChatCompletionsClient for each model.
client_configs = [
    {
        "name": model["name"],
        "client": ChatCompletionsClient(
            endpoint=model["endpoint"],
            credential=AzureKeyCredential(model["api_key"])
        )
    }
    for model in models
]

# Optionally, demonstrate using the clients by printing out their names and endpoints.
for config in client_configs:
    client = config["client"]
    print(f"Model: {config['name']} | Endpoint: {client._config.endpoint}")

Model: gpt-4o | Endpoint: https://openaiautoauthrtznwpq.openai.azure.com


In the clinical taxonomy, this type of problem often falls under the 'Administration and Workflow' category. This task focuses on key areas such as 'Prior Authorization' and 'Coverage Determinations', ensuring that the LLM effectively retrieves and scores the clinical relevance of a search query with respect to clinical documents.

Objective:
- Generate clinically relevant search queries.
- Retrieve and rank documents based on their alignment with clinical criteria (e.g., coverage requirements).

Evaluation Overview:
1. **Model Configuration:**
   - Utilize candidate LLMs (gpt-4o, o1, phi-4) via the Azure AI Foundry inference API.
   - Leverage environment-configured endpoints and credentials.
2. **Query Processing:**
   - Input a clinical search query to the models.
   - Generate responses that yield high-quality embeddings.
3. **Similarity Scoring:**
   - Use cosine similarity to compare query embeddings with document embeddings.
   - Rank documents based on clinical relevance.
4. **Clinical Impact:**
   - Evaluate model output against the MedHELM taxonomy criteria.
   - Provide actionable insights for real-world clinical applications.

This task ensures that the generated search query index aligns with clinical relevance standards critical to patient care workflows.


# Model Evaluation Strategy for Clinical Text Relevance

This document describes a hybrid evaluation framework for assessing the clinical relevance and quality of generative AI outputs when processing medical text. The evaluation combines **deterministic methods** (such as cosine similarity using domain-specific embeddings and n‑gram overlap metrics) with **LLM-assisted subjective evaluation** (using an LLM as a judge with the Azure AI Foundry Relevance metric).

## 1. Deterministic Evaluation Methods

Deterministic methods provide reproducible, objective measures of similarity between texts. In our setup, we highlight two custom evaluators:

- **BioBertSemanticSimilarity:**  
  Uses a domain-specific model (such as BioBERT or ClinicalBERT) to generate sentence embeddings. The cosine similarity between the embeddings of a query and a document is computed to determine semantic similarity.

- **NgramOverlapEvaluator:**  
  Uses an n‑gram based approach to compare the response against the ground truth. It calculates the Jaccard similarity between the sets of n‑grams extracted from each text.

### 1.1 Cosine Similarity with Domain-Specific Embeddings

Using models like **BioBERT** or **ClinicalBERT** to generate sentence embeddings provides a reproducible, deterministic method for assessing semantic similarity.

- **Embedding Extraction:**  
  The evaluator (i.e. `BioBertSemanticSimilarity`) tokenizes the input text and computes embeddings using a domain-specific BERT model. Typically, the `[CLS]` token or an average of the hidden states is used as the sentence representation.

- **Cosine Similarity Calculation:**  
  Compute the cosine similarity between the two fixed-length vectors:
  
  $begin:math:display$
  \\text{cosine\\_similarity}(\\mathbf{a}, \\mathbf{b}) = \\frac{\\mathbf{a} \\cdot \\mathbf{b}}{\\|\\mathbf{a}\\|\\|\\mathbf{b}\\|}
  $end:math:display$
  
  This score quantifies how semantically aligned the query and document are.

**Example snippet for BioBertSemanticSimilarity:**

```python
from dataclasses import dataclass
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from src.evals.custom.custom_evaluator import CustomEvaluator

@dataclass
class SemanticSimilarity:
    semantic_similarity: float

class BioBertSemanticSimilarity(CustomEvaluator):
    def __init__(self, model_name: str = "dmis-lab/biobert-base-cased-v1.1", **kwargs):
        super().__init__(**kwargs)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()  # Set model to evaluation mode
        for key, value in kwargs.items():
            setattr(self, key, value)

    def _clean_text(self, text: str) -> str:
        return " ".join(text.replace("\n", " ").replace("\t", " ").split())

    def _get_embedding(self, text: str) -> torch.Tensor:
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = self.model(**inputs)
        # Use the [CLS] token output as a sentence embedding.
        return outputs.last_hidden_state[:, 0, :]

    def __call__(self, *, response: str, ground_truth: str, **kwargs) -> SemanticSimilarity:
        try:
            clean_gt = self._clean_text(ground_truth)
            clean_resp = self._clean_text(response)
            emb_gt = self._get_embedding(clean_gt)
            emb_resp = self._get_embedding(clean_resp)
            sim = cosine_similarity(emb_gt.cpu().numpy(), emb_resp.cpu().numpy())[0][0]
        except Exception as e:
            sim = 0.0
        return SemanticSimilarity(semantic_similarity=sim)
```

### 1.2 N-gram Overlap Evaluator

The `NgramOverlapEvaluator` measures similarity between texts based on n‑gram overlap. It tokenizes the cleaned texts, generates a set of n‑grams (for example, bigrams), and then computes the Jaccard similarity as:

$begin:math:display$
\\text{Jaccard similarity} = \\frac{| \\text{intersection} |}{| \\text{union} |}
$end:math:display$

- **Text Preparation:**  
  Both texts are cleaned (removing punctuation and extra spaces) and optionally lowercased.
  
- **N-gram Extraction:**  
  The text is tokenized into words and n‑grams are generated based on the specified _n_ value.
  
- **Similarity Calculation:**  
  The similarity score is calculated using the Jaccard index between the n‑gram sets.

**Example snippet for NgramOverlapEvaluator:**

```python
from typing import TypedDict, List, Set
import re
from src.evals.custom.custom_evaluator import CustomEvaluator

class NgramSimilarityScore(TypedDict):
    ngram_similarity: float

class NgramOverlapEvaluator(CustomEvaluator):
    def __init__(self, n: int = 2, lowercase: bool = True, **kwargs):
        super().__init__(**kwargs)
        self.n = n
        self.lowercase = lowercase
        for key, value in kwargs.items():
            setattr(self, key, value)

    def _clean_text(self, text: str) -> str:
        if self.lowercase:
            text = text.lower()
        # Remove punctuation and collapse multiple spaces.
        text = re.sub(r"[^\w\s]", "", text)
        return " ".join(text.split())

    def _tokenize(self, text: str) -> List[str]:
        return text.split()

    def _get_ngrams(self, tokens: List[str]) -> Set[str]:
        ngrams = set()
        if len(tokens) < self.n:
            return ngrams
        for i in range(len(tokens) - self.n + 1):
            ngram = " ".join(tokens[i : i + self.n])
            ngrams.add(ngram)
        return ngrams

    def __call__(self, *, response: str, ground_truth: str, **kwargs) -> NgramSimilarityScore:
        try:
            clean_gt = self._clean_text(ground_truth)
            clean_resp = self._clean_text(response)
            tokens_gt = self._tokenize(clean_gt)
            tokens_resp = self._tokenize(clean_resp)
            ngrams_gt = self._get_ngrams(tokens_gt)
            ngrams_resp = self._get_ngrams(tokens_resp)
            if not ngrams_gt and not ngrams_resp:
                score = 1.0
            else:
                intersection = ngrams_gt.intersection(ngrams_resp)
                union = ngrams_gt.union(ngrams_resp)
                score = len(intersection) / len(union) if union else 0.0
        except Exception:
            score = 0.0
        return {"ngram_similarity": score}
```

## 2. LLM-Assisted Evaluation

In addition to these deterministic methods, LLM-assisted evaluation uses a model acting as a judge (for example, via Azure AI Foundry's RelevanceEvaluator) to assess the quality of model outputs. This approach provides subjective insights similar to human judgment, capturing nuances that strict numerical metrics may miss.

## 3. Hybrid Evaluation Workflow

To create a robust evaluation pipeline:
1. **Deterministic Evaluation:**  
   - Use `BioBertSemanticSimilarity` to compute cosine similarity and capture semantic alignment.
   - Use `NgramOverlapEvaluator` to compute Jaccard similarity based on n-gram overlap.
2. **LLM-Assisted Evaluation:**  
   - Use the RelevanceEvaluator to generate a human-like relevance score.
3. **Aggregation:**  
   - Combine deterministic scores with the LLM-assisted relevance score to obtain a final measure of clinical relevance and quality.

This hybrid evaluation approach ensures that both objective (deterministic) measures and nuanced (LLM-assisted) judgments are considered for a thorough assessment of clinical text relevance.

---

*This document serves as a guideline for setting up and integrating these deterministic evaluators with LLM-assisted methodologies to achieve a comprehensive and reproducible evaluation framework for clinical text relevance.*


In [3]:
import importlib
import inspect
import json
import logging
import os
import shutil
import subprocess
from abc import ABC, abstractmethod
from typing import List, Tuple, final

import yaml
from azure.ai.evaluation import evaluate, RelevanceEvaluator
from dotenv import load_dotenv

# @TODO: Remove this import when the package fix is available.
from azure.ai.evaluation._evaluate._eval_run import EvalRun
import src.evals.sdk.custom_azure_ai_evaluations as custom_eval
from src.aifoundry.aifoundry_helper import AIFoundryManager
from src.evals.sdk.custom_azure_ai_evaluations import custom_start_run
from src.pipeline.utils import load_config
from src.utils.ml_logging import get_logger

# Override EvalRun._start_run with our custom start run.
EvalRun._start_run = custom_start_run

logger = get_logger("MedIndexerEvaluator", "INFO")

# Instantiate our custom evaluators for deterministic similarity.
from src.evals.custom.ngram_similarity import NgramSimilarityEvaluator
from src.evals.custom.semantic_similarity import SemanticSimilarityEvaluator

# These evaluators remain constant across runs.
ngram_eval = NgramSimilarityEvaluator(n=2)
semantic_eval = SemanticSimilarityEvaluator(model_name="dmis-lab/biobert-base-cased-v1.1")

# Load corpus and queries from JSONL files.
root = os.getcwd()
corpus_file = f"../evals/benchmark/medindexer/corpus.jsonl"
queries_file = f"../evals/benchmark/medindexer/queries.jsonl"

corpus = []
with open(corpus_file, "r", encoding="utf8") as f:
    for line in f:
        try:
            data = json.loads(line)
            if "chunk" in data:
                corpus.append(data["chunk"])
        except Exception as e:
            logger.error(f"Error reading corpus line: {e}")

queries = []
with open(queries_file, "r", encoding="utf8") as f:
    for line in f:
        try:
            data = json.loads(line)
            if "query" in data:
                queries.append(data["query"])
        except Exception as e:
            logger.error(f"Error reading queries line: {e}")

# Generate the evaluation dataset: each query paired with each chunk.
evaluation_data = []
for query in queries:
    for chunk in corpus:
        evaluation_data.append({
            "query": query,
            "ground_truth": chunk,
            "response": chunk
        })

/Users/marcjimz/Documents/Development/aihlsignited-medevals/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Write the evaluation dataset to a temporary JSONL file.
temp_dataset = "temp_medindexer_dataset.jsonl"
with open(temp_dataset, "w", encoding="utf8") as f:
    for item in evaluation_data[:100]:  # Limit to first 100 items for testing
        f.write(json.dumps(item) + "\n")

# Function to generate custom tags.
def generate_custom_tags(case_id: str, git_commit: str, class_name: str, model_name: str) -> List[Tuple[str, str]]:
    return [
        ("case", case_id),
        ("commit", git_commit),
        ("LLM_model", model_name)
    ]

# Retrieve current git commit hash.
try:
    git_hash = (
        subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.STDOUT
        )
        .decode()
        .strip()
    )
except Exception:
    git_hash = "unknown"

# Instantiate the AIFoundryManager to obtain project configuration.
ai_foundry_manager = AIFoundryManager()

# Prepare a dictionary to collect results for each model configuration.
all_results = {}

# For each model configuration, run an evaluation run.
# Build the model_config for the RelevanceEvaluator.
model_config = {
    "azure_endpoint": models[0]["endpoint"],
    "api_key": models[0]["api_key"],
    "azure_deployment": os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT_ID"),
    "api_version": os.environ.get("AZURE_OPENAI_API_VERSION"),
}
    
# Instantiate the RelevanceEvaluator with this model config.
relevance_eval = RelevanceEvaluator(model_config)
    
# Combine evaluators: semantic and ngram remain same; update relevance evaluator.
evaluators = {
    "semantic": semantic_eval,
    "ngram": ngram_eval,
    "relevance": relevance_eval,
}
    
# Generate custom tags using this model's name.
custom_tags = generate_custom_tags("medindexer_test", git_hash, "PipelineEvaluator", models[0]["name"])
custom_eval.CUSTOM_TAGS = custom_tags

# Build evaluator configuration for column mapping.
evaluator_config = {
    "semantic": {"column_mapping": {"response": "${data.response}", "ground_truth": "${data.ground_truth}"}},
    "ngram": {"column_mapping": {"response": "${data.response}", "ground_truth": "${data.ground_truth}"}},
    "relevance": {"column_mapping": {"response": "${data.response}", "query": "${data.query}"}}
}

# Define a unique evaluation submission name per model.
evaluation_name = f"MedIndexer_Evaluation_Submission_{models[0]['name']}"

# Run evaluation using the Azure AI Evaluation SDK.
results = evaluate(
    evaluation_name=evaluation_name,
    data=temp_dataset,
    evaluators=evaluators,
    evaluator_config=evaluator_config,
    azure_ai_project=ai_foundry_manager.project_config,
)


[2025-04-08 21:12:58 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-08 21:12:58 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-08 21:12:58 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-08 21:12:58 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_common_base_eval_asyncevaluatorbase_4o3xzuqm_20250408_211258_834489, log path: /Users/marcjimz/.promptflow/.runs/azure_ai_evaluation_evaluators_common_base_eval_asyncevaluatorbase_4o3xzuqm_20250408_211258_834489/logs.txt
[2025-04-08 21:12:58 -0700][promptflow._sdk._orchestrator.

2025-04-08 21:13:03 -0700   78134 execution.bulk     INFO     Finished 10 / 100 lines.
2025-04-08 21:13:03 -0700   78134 execution.bulk     INFO     Average execution time for completed lines: 0.46 seconds. Estimated time for incomplete lines: 41.4 seconds.
2025-04-08 21:13:06 -0700   78134 execution.bulk     INFO     Finished 20 / 100 lines.
2025-04-08 21:13:07 -0700   78134 execution.bulk     INFO     Average execution time for completed lines: 0.41 seconds. Estimated time for incomplete lines: 32.8 seconds.
2025-04-08 21:13:10 -0700   78134 execution.bulk     INFO     Finished 30 / 100 lines.
2025-04-08 21:13:10 -0700   78134 execution.bulk     INFO     Average execution time for completed lines: 0.38 seconds. Estimated time for incomplete lines: 26.6 seconds.
2025-04-08 21:13:13 -0700   78134 execution.bulk     INFO     Finished 40 / 100 lines.
2025-04-08 21:13:13 -0700   78134 execution.bulk     INFO     Average execution time for completed lines: 0.36 seconds. Estimated time for 

[2025-04-08 21:13:17 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: RateLimitError: Error code: 429 - {'error': {'code': '429', 'message': 'Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2025-01-01-preview have exceeded token rate limit of your current OpenAI S0 pricing tier. Please retry after 43 seconds. Please go here: https://aka.ms/oai/quotaincrease if you would like to further increase the default rate limit. For Free Account customers, upgrade to Pay as you Go here: https://aka.ms/429TrialUpgrade.'}}
[2025-04-08 21:13:17 -0700][promptflow.core._prompty_utils][WARNING] - RateLimitError #0, Retry-After=43, Back off 43.0 seconds for retry.
[2025-04-08 21:13:17 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: RateLimitError: Error code: 429 - {'error': {'code': '429', 'message': 'Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2025-01-01-preview have exceeded token rate limit of your

2025-04-08 21:13:17 -0700   78134 execution.bulk     INFO     Process name(SpawnProcess-29)-Process id(78728)-Line number(61) completed.
2025-04-08 21:13:17 -0700   78134 execution.bulk     INFO     Process name(SpawnProcess-29)-Process id(78728)-Line number(65) start execution.
2025-04-08 21:13:17 -0700   78134 execution.bulk     INFO     Process name(SpawnProcess-29)-Process id(78728)-Line number(65) completed.
2025-04-08 21:13:17 -0700   78134 execution.bulk     INFO     Process name(SpawnProcess-29)-Process id(78728)-Line number(66) start execution.
2025-04-08 21:13:17 -0700   78134 execution.bulk     INFO     Process name(SpawnProcess-28)-Process id(78724)-Line number(64) completed.
2025-04-08 21:13:17 -0700   78134 execution.bulk     INFO     Process name(SpawnProcess-28)-Process id(78724)-Line number(67) start execution.
2025-04-08 21:13:17 -0700   78134 execution.bulk     INFO     Process name(SpawnProcess-30)-Process id(78735)-Line number(59) completed.
2025-04-08 21:13:17 -07

[2025-04-08 21:13:17 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: RateLimitError: Error code: 429 - {'error': {'code': '429', 'message': 'Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2025-01-01-preview have exceeded token rate limit of your current OpenAI S0 pricing tier. Please retry after 42 seconds. Please go here: https://aka.ms/oai/quotaincrease if you would like to further increase the default rate limit. For Free Account customers, upgrade to Pay as you Go here: https://aka.ms/429TrialUpgrade.'}}
[2025-04-08 21:13:17 -0700][promptflow.core._prompty_utils][WARNING] - RateLimitError #0, Retry-After=42, Back off 42.0 seconds for retry.


...

2025-04-08 21:14:02 -0700   78134 execution.bulk     INFO     Finished 70 / 100 lines.
2025-04-08 21:14:02 -0700   78134 execution.bulk     INFO     Average execution time for completed lines: 0.91 seconds. Estimated time for incomplete lines: 27.3 seconds.


[2025-04-08 21:14:04 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: RateLimitError: Error code: 429 - {'error': {'code': '429', 'message': 'Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2025-01-01-preview have exceeded token rate limit of your current OpenAI S0 pricing tier. Please retry after 1 second. Please go here: https://aka.ms/oai/quotaincrease if you would like to further increase the default rate limit. For Free Account customers, upgrade to Pay as you Go here: https://aka.ms/429TrialUpgrade.'}}
[2025-04-08 21:14:04 -0700][promptflow.core._prompty_utils][WARNING] - RateLimitError #0, Retry-After=1, Back off 1.0 seconds for retry.
[2025-04-08 21:14:04 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: RateLimitError: Error code: 429 - {'error': {'code': '429', 'message': 'Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2025-01-01-preview have exceeded token rate limit of your cur

2025-04-08 21:14:05 -0700   78134 execution.bulk     INFO     Finished 80 / 100 lines.
2025-04-08 21:14:05 -0700   78134 execution.bulk     INFO     Average execution time for completed lines: 0.83 seconds. Estimated time for incomplete lines: 16.6 seconds.


[2025-04-08 21:14:05 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: RateLimitError: Error code: 429 - {'error': {'code': '429', 'message': 'Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2025-01-01-preview have exceeded token rate limit of your current OpenAI S0 pricing tier. Please retry after 1 second. Please go here: https://aka.ms/oai/quotaincrease if you would like to further increase the default rate limit. For Free Account customers, upgrade to Pay as you Go here: https://aka.ms/429TrialUpgrade.'}}
[2025-04-08 21:14:05 -0700][promptflow.core._prompty_utils][WARNING] - RateLimitError #0, Retry-After=1, Back off 1.0 seconds for retry.
[2025-04-08 21:14:07 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: RateLimitError: Error code: 429 - {'error': {'code': '429', 'message': 'Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2025-01-01-preview have exceeded token rate limit of your cur

2025-04-08 21:14:08 -0700   78134 execution.bulk     INFO     Finished 90 / 100 lines.
2025-04-08 21:14:08 -0700   78134 execution.bulk     INFO     Average execution time for completed lines: 0.77 seconds. Estimated time for incomplete lines: 7.7 seconds.
2025-04-08 21:14:12 -0700   78134 execution.bulk     INFO     Finished 100 / 100 lines.
2025-04-08 21:14:12 -0700   78134 execution.bulk     INFO     Average execution time for completed lines: 0.73 seconds. Estimated time for incomplete lines: 0.0 seconds.


In [ ]:
<pre>
# SECTION 4: Evaluators

"""
We will demonstrate two approaches to evaluate Relevance:

1) Deterministic approach:
   - For instance, a string overlap or BM25 similarity-based approach, or a simple function that checks how many domain-specific keywords appear in the chunk vs. the query.

2) LLM-as-a-Judge approach:
   - Use a dedicated LLM call that, given the query and chunk content, rates the relevance from 1 to 5, and optionally provides a rationale ("thought_chain").

We’ll show code snippets for each approach. Typically, both approaches are run, and the final score might be an average or a direct comparison between the two methods.
"""
</pre>


In [ ]:
<pre><code>
import re

##############################
# 1. Deterministic Approach  #
##############################
def deterministic_relevance(query: str, chunk: str) -> float:
    """
    A simple, naive scoring function: 
    - Count the frequency of relevant words from the query in the chunk.
    - Scale from 0 to 5.
    """
    query_tokens = re.findall(r"\w+", query.lower())
    chunk_text = chunk.lower()
    count = sum(token in chunk_text for token in query_tokens)
    # Score from 1 to 5 based on how many tokens matched
    # This is naive; you might instead use BM25 or a more robust similarity measure
    return min(5, max(1, count))  # ensure in [1..5]

##############################
# 2. LLM as a Judge Approach #
##############################
def llm_judge_relevance(query: str, chunk: str, llm_client=None) -> float:
    """
    Hypothetical function that calls an LLM to evaluate relevance.
    We'll mock a response for demonstration.
    In production, you'd pass 'query' & 'chunk' to your GPT-4o or phi-4 endpoint and parse a structured JSON answer.
    """
    # Example system or user prompt might be: 
    #   "You are evaluating how relevant the chunk of text is to the query. 
    #    Return a numeric value 1-5, plus an explanation"
    # Here we simulate a random or fixed response for demonstration.
    simulated_score = 4  # placeholder
    return simulated_score

# Let's create a function to run both evaluations
def evaluate_queries(queries, chunks):
    results = []
    for q in queries:
        # Filter chunks that share the same doc title, if relevant
        relevant_chunks = [c for c in chunks if c["title"] == q["title"]]

        for c in relevant_chunks:
            # Deterministic approach
            det_score = deterministic_relevance(q["query"], c["chunk"])
            # LLM judge approach
            llm_score = llm_judge_relevance(q["query"], c["chunk"])

            results.append({
                "query_id": q["id"],
                "query_text": q["query"],
                "document_id": c["id"],
                "deterministic_score": det_score,
                "llm_judge_score": llm_score
            })
    return results

# Evaluate the queries
scores = evaluate_queries(queries, chunks)
scores
</code></pre>


In [ ]:
<pre>
# SECTION 5: Generating a Leaderboard (via Azure AI Foundry)

"""
Now we assume we want to compare 3 or 4 different LLMs on the same set of queries & chunks. 
We can store their results and use Azure AI Foundry to produce a consolidated leaderboard.

In a real scenario, each model might produce a 'response' (i.e., chunk retrieval or generated text). 
We then feed these responses into the Foundry's Evaluate API with relevant metrics (NDCG, Precision@K, or in this case, a custom Relevance scoring approach).

Below is pseudo-code showing how we might do so using the 'eval.py' and 'BenchmarkOrchestrator' style from the user-provided example. 
We'll do a simple print-out of average scores as a miniature leaderboard.
"""
</pre>


In [ ]:
<pre>
# Example Output Explanation

"""
The above `leaderboard` might look something like:

  query_id | deterministic_score | llm_judge_score | average_score
  -------- | ------------------- | --------------- | -------------
     q1    |        3.5         |       4.0       |      3.75
     q2    |        2.0         |       3.0       |      2.50
     ...    ...                  ...               ...

We can then compare these results across multiple candidate LLMs to see which yields the highest average Relevance scores. 
For a final step, you might store these in Azure AI Foundry, do a custom ranking, or share a combined chart of “Deterministic vs. LLM Judge” scores.
"""
</pre>


In [ ]:
<pre>
# Conclusion

"""
**Summary of Steps**:
1. **Candidate LLMs**: We used placeholders for GPT-4o, o1, phi-3, phi-4.
2. **Task Category**: "Administration & Workflow" (Prior Authorization queries) from the MedHELM taxonomy.
3. **Evaluation**: Measured Relevance using two approaches:
   - Deterministic (token overlap or other lexical measure).
   - LLM-as-a-Judge (model self-scores the chunk’s relevance).
4. **Evaluators**: Demonstrated code for 2 out of 3 possible evaluators (the third could be a final human evaluator).
5. **Leaderboard**: Showed a simplified approach to producing a consolidated scoreboard. 
   This can be integrated with Azure AI Foundry for advanced metrics (e.g., NDCG@k, MAP, etc.).

By following these steps, healthcare organizations can more systematically measure whether a language model accurately identifies relevant content in a clinical policy or prior authorization document. This approach is in line with the broader MedHELM framework, emphasizing real-world use cases, multi-metric measurement, and standardized reporting.
"""
</pre>
